In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from scipy.stats import randint, uniform


train_x = pd.read_csv("/content/airbnb_train_x (4).csv")
train_y = pd.read_csv("/content/airbnb_train_y (4).csv")
#test_x = pd.read_csv("/content/airbnb_test_x (4).csv")


In [2]:
numeric_cols = train_x.select_dtypes(include=['float64', 'int64']).columns
missing_numeric = train_x[numeric_cols].isnull().sum()
missing_numeric = missing_numeric[missing_numeric > 0]
print(missing_numeric)
train_x.shape

host_response_rate           13450
host_total_listings_count      169
accommodates                    50
bathrooms                      281
bedrooms                       113
beds                            91
square_feet                  90387
price                          533
weekly_price                 73556
monthly_price                69637
security_deposit             49157
cleaning_fee                 20180
dtype: int64


(92067, 61)

In [3]:
# Drop square_feet, weekly_price, monthly_price
# Drop high-missing or redundant columns
train_x.drop(columns=['square_feet', 'weekly_price', 'monthly_price'], inplace=True)

# Fill optional fees with 0
train_x['cleaning_fee'] = train_x['cleaning_fee'].fillna(0)
train_x['security_deposit'] = train_x['security_deposit'].fillna(0)

# Handle host_response_rate
train_x['host_response_rate'] = train_x['host_response_rate'].astype(str).str.replace('%', '').replace('nan', np.nan).astype(float)
train_x['host_response_rate'] = train_x['host_response_rate'].fillna(train_x['host_response_rate'].median())

# Drop rows with missing essential numeric fields
train_x.dropna(subset=['price', 'accommodates', 'bedrooms', 'beds', 'bathrooms'], inplace=True)

# Sync target variable to match filtered rows in train_x
train_y = train_y.loc[train_x.index]
train_x.reset_index(drop=True, inplace=True)
train_y.reset_index(drop=True, inplace=True)

In [4]:
pd.set_option('display.max_columns', None)

# Load and subset your data
df_subset = train_x.head(150)

# Select and display only categorical columns
categorical_columns = df_subset.select_dtypes(include='object')
print(categorical_columns.head(1).T)

                                                                       0
name                                 Perfect Apt Steps From Central Park
summary                Central Park Apt! 2 bedroom 1 bath, a block aw...
space                  2 bedroom 1 bath, a block away from train, 1 b...
description            Central Park Apt! 2 bedroom 1 bath, a block aw...
experiences_offered                                                 none
neighborhood_overview  Central Park North is the north side of Centra...
notes                                                                NaN
transit                The apartment is just a block away from the 2/...
access                 You'll have access to the common areas: living...
interaction            I'll be available via text throughout your sta...
house_rules            - Not suitable for pets - No parties or events...
host_name                                                          Riley
host_since                                         

In [5]:
# Step 1: Fill missing values
train_x['borough'] = train_x['street'].fillna('Unknown')

# Step 2: Extract the first token before the first comma (usually neighborhood/borough)
train_x['borough'] = train_x['borough'].apply(lambda x: x.split(',')[0].strip())

# Step 3 (optional): Group rare boroughs with less than 10 listings into 'Other'
borough_counts = train_x['borough'].value_counts()
common_boroughs = borough_counts[borough_counts >= 8].index

train_x['borough'] = train_x['borough'].apply(lambda x: x if x in common_boroughs else 'Other')

# Step 4 (optional): Label encode for modeling
from sklearn.preprocessing import LabelEncoder
le_borough = LabelEncoder()
train_x['borough'] = le_borough.fit_transform(train_x['borough'])

train_x.drop(columns='street', inplace=True)


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# List of text columns
text_cols = [
    'name', 'summary', 'space', 'description', 'experiences_offered',
    'neighborhood_overview', 'notes', 'transit', 'access',
    'interaction', 'house_rules'
]

# TF-IDF vectorization with handling for empty columns
for col in text_cols:
    train_x[col] = train_x[col].fillna('').astype(str).str.strip()

    # Skip column if it's empty after cleaning
    if train_x[col].str.len().sum() == 0:
        print(f"Skipping TF-IDF for column: {col} (empty after cleaning)")
        train_x.drop(columns=col, inplace=True)
        continue

    # Initialize vectorizer
    tfidf = TfidfVectorizer(stop_words='english', max_features=50)

    try:
        tfidf_matrix = tfidf.fit_transform(train_x[col])
        tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=[f'{col}_tfidf_{word}' for word in tfidf.get_feature_names_out()])
        tfidf_df.index = train_x.index
        train_x = pd.concat([train_x.drop(columns=col), tfidf_df], axis=1)
    except ValueError:
        print(f"Skipping column '{col}' due to empty vocabulary.")
        train_x.drop(columns=col, inplace=True)


Skipping column 'experiences_offered' due to empty vocabulary.


In [7]:
# Step 1: Handle missing values and lowercase everything
train_x['host_verifications'] = train_x['host_verifications'].fillna('').str.lower()

# Step 2: Convert comma-separated string to list
train_x['host_verifications'] = train_x['host_verifications'].apply(lambda x: [i.strip() for i in x.split(',') if i.strip()])

# Step 3: Apply MultiLabelBinarizer to create binary features
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
host_ver_df = pd.DataFrame(mlb.fit_transform(train_x['host_verifications']),
                           columns=[f'hostverif_{col}' for col in mlb.classes_],
                           index=train_x.index)

# Step 4: Drop original and concat new columns
train_x.drop(columns='host_verifications', inplace=True)
train_x = pd.concat([train_x, host_ver_df], axis=1)

In [8]:
from sklearn.preprocessing import LabelEncoder

# Combine year and month
train_x['host_since'] = pd.to_datetime(train_x['host_since'], errors='coerce')
train_x['host_since'] = train_x['host_since'].dt.to_period('M').astype(str)

# Label encode
le = LabelEncoder()
train_x['host_since'] = le.fit_transform(train_x['host_since'])

In [9]:
# Remove '%' and convert to float
train_x['host_acceptance_rate'] = train_x['host_acceptance_rate'].str.rstrip('%').astype(float)

In [10]:
response_time_order = {
    'within an hour': 0,
    'within a few hours': 1,
    'within a day': 2,
    'a few days or more': 3
}

train_x['host_response_time'] = train_x['host_response_time'].map(response_time_order)


In [11]:
train_x.drop(columns='neighborhood', inplace=True)

In [12]:
train_x.drop(columns='host_about', inplace=True)

In [13]:
train_x.drop(columns='host_location', inplace=True)

In [14]:
train_x.drop(columns='host_neighbourhood', inplace=True)

In [15]:
# Step 1: Standardize formatting
train_x['neighborhood_group'] = train_x['neighborhood_group'].astype(str).str.strip().str.title()

# Step 2: Label encode
from sklearn.preprocessing import LabelEncoder
le_ngroup = LabelEncoder()
train_x['neighborhood_group'] = le_ngroup.fit_transform(train_x['neighborhood_group'])

In [16]:
# Step 1: Clean up formatting
train_x['city'] = train_x['city'].astype(str).str.strip().str.title()

# Step 2: Group cities with very low frequency (e.g., < 20) into "Other"
city_counts = train_x['city'].value_counts()
valid_cities = city_counts[city_counts >= 20].index

train_x['city'] = train_x['city'].apply(lambda x: x if x in valid_cities else 'Other')

# Step 3: Label encode
from sklearn.preprocessing import LabelEncoder
le_city = LabelEncoder()
train_x['city'] = le_city.fit_transform(train_x['city'])

In [17]:
# Step 1: Ensure ZIP is string
train_x['zipcode'] = train_x['zipcode'].astype(str)

# Step 2: Extract first 2 digits as region
train_x['zipcode'] = train_x['zipcode'].str[:2]

# Step 3: Label encode the 2-digit ZIP region
from sklearn.preprocessing import LabelEncoder
le_zip2 = LabelEncoder()
train_x['zipcode'] = le_zip2.fit_transform(train_x['zipcode'])

In [18]:
train_x['state'] = train_x['state'].astype(str).str.strip().str.upper()
from sklearn.preprocessing import LabelEncoder
le_state = LabelEncoder()
train_x['state'] = le_state.fit_transform(train_x['state'])

In [19]:
import pandas as pd

train_x['first_review'] = pd.to_datetime(train_x['first_review'], errors='coerce')
train_x['host_since'] = pd.to_datetime(train_x['host_since'], errors='coerce')
print(train_x[['first_review', 'host_since']].dtypes)

first_review    datetime64[ns]
host_since      datetime64[ns]
dtype: object


In [20]:
print(train_x.columns.tolist())


['host_name', 'host_since', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_listings_count', 'host_total_listings_count', 'neighborhood_group', 'city', 'state', 'zipcode', 'market', 'smart_location', 'country_code', 'country', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'bed_type', 'amenities', 'price', 'security_deposit', 'cleaning_fee', 'guests_included', 'extra_people', 'minimum_nights', 'maximum_nights', 'availability_30', 'availability_60', 'availability_90', 'availability_365', 'first_review', 'license', 'jurisdiction_names', 'cancellation_policy', 'features', 'borough', 'name_tfidf_1br', 'name_tfidf_apartment', 'name_tfidf_apt', 'name_tfidf_bath', 'name_tfidf_beach', 'name_tfidf_beautiful', 'name_tfidf_bed', 'name_tfidf_bedroom', 'name_tfidf_br', 'name_tfidf_bright', 'name_tfidf_brooklyn', 'name_tfidf_central', 'name_tfidf_charming', 'name_tfidf_city', 'name_tfidf_clean', 'name_tfidf_close', 'n

In [21]:
# Step 1: Fill missing values
train_x['smart_location'] = train_x['smart_location'].fillna('unknown')

# Step 2: Standardize formatting (lowercase everything for consistency)
train_x['smart_location'] = train_x['smart_location'].str.strip().str.lower()

# Step 3: Extract major city and state (first 2 tokens)
train_x['smart_location'] = train_x['smart_location'].apply(
    lambda x: ', '.join(x.split(',')[:2]) if ',' in x else x
)

# Step 4: Group rare values as 'other' if they appear < 20 times
location_counts = train_x['smart_location'].value_counts()
valid_locations = location_counts[location_counts >= 20].index

train_x['smart_location'] = train_x['smart_location'].apply(
    lambda x: x if x in valid_locations else 'other'
)

# Step 5: Label encode
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
train_x['smart_location'] = le.fit_transform(train_x['smart_location'])

In [22]:
train_x['smart_location'].value_counts()

,count
smart_location,
55,13275
47,13208
14,10768
73,5898
7,5168
...,...
5,21
19,20
60,20


In [23]:
threshold = 10

# Count frequencies
property_counts = train_x['property_type'].value_counts()

# Replace rare types with 'Other'
train_x['property_type'] = train_x['property_type'].apply(
    lambda x: x if pd.notna(x) and property_counts.get(x, 0) >= 10 else 'Other')
train_x['property_type'].value_counts()

,count
property_type,
Apartment,51452
House,28931
Condominium,3143
Townhouse,1912
Loft,1609
Other,786
Bed & Breakfast,752
Guesthouse,693
Bungalow,538


In [24]:
from sklearn.preprocessing import LabelEncoder

# Columns to encode
cols_to_encode = ['country_code', 'country', 'property_type', 'room_type', 'bed_type']

# Apply Label Encoding
le = LabelEncoder()
for col in cols_to_encode:
    train_x[col] = train_x[col].fillna('Unknown')         # Handle missing values
    train_x[col] = train_x[col].astype(str)               # Ensure type is string
    train_x[col] = le.fit_transform(train_x[col])         # Encode

In [25]:
from sklearn.preprocessing import MultiLabelBinarizer
import pandas as pd

# Step 1: Clean and standardize the 'amenities' column
train_x['amenities'] = train_x['amenities'].fillna('').str.lower().str.replace(r'[^a-z0-9, ]', '', regex=True)

# Step 2: Convert to list
train_x['amenities'] = train_x['amenities'].apply(lambda x: [i.strip() for i in x.split(',') if i.strip()])

# Step 3: Multi-hot encode
mlb = MultiLabelBinarizer()
amenities_encoded = pd.DataFrame(mlb.fit_transform(train_x['amenities']),
                                 columns=mlb.classes_,
                                 index=train_x.index)

# Step 4: Replace original 'amenities' column with the binary columns
train_x.drop(columns='amenities', inplace=True)
train_x = pd.concat([train_x, amenities_encoded], axis=1)
train_x.head()

,host_name,host_since,host_response_time,host_response_rate,host_acceptance_rate,host_listings_count,host_total_listings_count,neighborhood_group,city,state,zipcode,market,smart_location,country_code,country,latitude,longitude,property_type,room_type,accommodates,bathrooms,bedrooms,beds,bed_type,price,security_deposit,cleaning_fee,guests_included,extra_people,minimum_nights,maximum_nights,availability_30,availability_60,availability_90,availability_365,first_review,license,jurisdiction_names,cancellation_policy,features,borough,name_tfidf_1br,name_tfidf_apartment,name_tfidf_apt,name_tfidf_bath,name_tfidf_beach,name_tfidf_beautiful,name_tfidf_bed,name_tfidf_bedroom,name_tfidf_br,name_tfidf_bright,name_tfidf_brooklyn,name_tfidf_central,name_tfidf_charming,name_tfidf_city,name_tfidf_clean,name_tfidf_close,name_tfidf_condo,name_tfidf_cottage,name_tfidf_cozy,name_tfidf_downtown,name_tfidf_east,name_tfidf_garden,name_tfidf_great,name_tfidf_heart,name_tfidf_hill,name_tfidf_hollywood,name_tfidf_home,name_tfidf_house,name_tfidf_large,name_tfidf_location,name_tfidf_loft,name_tfidf_luxury,name_tfidf_manhattan,name_tfidf_modern,name_tfidf_near,name_tfidf_new,name_tfidf_park,name_tfidf_parking,name_tfidf_private,name_tfidf_quiet,name_tfidf_room,name_tfidf_spacious,name_tfidf_studio,name_tfidf_suite,name_tfidf_sunny,name_tfidf_view,name_tfidf_village,name_tfidf_walk,name_tfidf_west,name_tfidf_williamsburg,summary_tfidf_access,summary_tfidf_apartment,summary_tfidf_area,summary_tfidf_away,summary_tfidf_bars,summary_tfidf_bathroom,summary_tfidf_beach,summary_tfidf_beautiful,summary_tfidf_bed,summary_tfidf_bedroom,summary_tfidf_blocks,summary_tfidf_business,summary_tfidf_city,summary_tfidf_close,summary_tfidf_couples,summary_tfidf_distance,summary_tfidf_downtown,summary_tfidf_enjoy,summary_tfidf_floor,summary_tfidf_good,summary_tfidf_great,summary_tfidf_heart,summary_tfidf_home,summary_tfidf_house,summary_tfidf_just,summary_tfidf_kitchen,summary_tfidf_large,summary_tfidf_living,summary_tfidf_ll,summary_tfidf_located,summary_tfidf_location,summary_tfidf_love,summary_tfidf_minutes,summary_tfidf_neighborhood,summary_tfidf_new,summary_tfidf_park,summary_tfidf_parking,summary_tfidf_perfect,summary_tfidf_place,summary_tfidf_private,summary_tfidf_queen,summary_tfidf_quiet,summary_tfidf_restaurants,summary_tfidf_room,summary_tfidf_space,summary_tfidf_spacious,summary_tfidf_street,summary_tfidf_travelers,summary_tfidf_walk,summary_tfidf_walking,space_tfidf_access,space_tfidf_apartment,space_tfidf_area,space_tfidf_available,space_tfidf_away,space_tfidf_bath,space_tfidf_bathroom,space_tfidf_beautiful,space_tfidf_bed,space_tfidf_bedroom,space_tfidf_bedrooms,space_tfidf_building,space_tfidf_city,space_tfidf_clean,space_tfidf_coffee,space_tfidf_comfortable,space_tfidf_dining,space_tfidf_downtown,space_tfidf_enjoy,space_tfidf_floor,space_tfidf_free,space_tfidf_fully,space_tfidf_great,space_tfidf_guests,space_tfidf_high,space_tfidf_home,space_tfidf_house,space_tfidf_just,space_tfidf_kitchen,space_tfidf_large,space_tfidf_living,space_tfidf_located,space_tfidf_minutes,space_tfidf_neighborhood,space_tfidf_new,space_tfidf_park,space_tfidf_parking,space_tfidf_place,space_tfidf_private,space_tfidf_queen,space_tfidf_quiet,space_tfidf_restaurants,space_tfidf_room,space_tfidf_size,space_tfidf_space,space_tfidf_spacious,space_tfidf_stay,space_tfidf_street,space_tfidf_tv,space_tfidf_walk,description_tfidf_access,description_tfidf_apartment,description_tfidf_area,description_tfidf_available,description_tfidf_away,description_tfidf_bars,description_tfidf_bathroom,description_tfidf_beautiful,description_tfidf_bed,description_tfidf_bedroom,description_tfidf_blocks,description_tfidf_building,description_tfidf_city,description_tfidf_close,description_tfidf_coffee,description_tfidf_comfortable,description_tfidf_distance,description_tfidf_downtown,description_tfidf_enjoy,description_tfidf_floor,description_tfidf_free,description_tfidf_great,description_tfidf_guests,description_t

In [26]:
## for feature

import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

# Step 1: Clean and standardize the 'features' column
train_x['features'] = train_x['features'].fillna('').str.replace(r'[^a-zA-Z0-9, &]', '', regex=True)

# Step 2: Convert to list
train_x['features'] = train_x['features'].apply(lambda x: [i.strip() for i in x.split(',') if i.strip()])

# Step 3: Multi-hot encode into binary columns
mlb = MultiLabelBinarizer()
features_encoded = pd.DataFrame(mlb.fit_transform(train_x['features']),
                                columns=mlb.classes_,
                                index=train_x.index)

# Step 4: Drop original and join new binary features
train_x.drop(columns='features', inplace=True)
train_x = pd.concat([train_x, features_encoded], axis=1)

In [27]:
train_x.drop(columns='license', inplace=True)


In [28]:
from sklearn.preprocessing import OrdinalEncoder

# Ordered by strictness
policy_order = [['flexible', 'moderate', 'strict', 'super_strict_30', 'super_strict_60', 'no_refunds']]
oe = OrdinalEncoder(categories=policy_order)

train_x['cancellation_policy'] = train_x[['cancellation_policy']].fillna('flexible')  # Fill missing with baseline
train_x['cancellation_policy'] = oe.fit_transform(train_x[['cancellation_policy']])

In [29]:
# Step 1: Fill NaN with 'Unknown' and standardize
train_x['jurisdiction_names'] = train_x['jurisdiction_names'].fillna('Unknown').str.upper()

# Step 2: Get count of values
counts = train_x['jurisdiction_names'].value_counts()

# Step 3: Create a mask of jurisdictions that appear less than 10 times
rare_names = counts[counts < 10].index

# Step 4: Replace rare values with 'Other'
train_x['jurisdiction_names'] = train_x['jurisdiction_names'].apply(
    lambda x: 'Other' if x in rare_names else x
)

# Step 5: Label encode
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
train_x['jurisdiction_names'] = le.fit_transform(train_x['jurisdiction_names'])

In [30]:
# Step 1: Fill missing if any
train_x['market'] = train_x['market'].fillna('Unknown')

# Step 2: Group rare markets (those with count < 8) as 'Other'
market_counts = train_x['market'].value_counts()
valid_markets = market_counts[market_counts >= 8].index

train_x['market'] = train_x['market'].apply(
    lambda x: x if x in valid_markets else 'Other'
)
# Step 3: Label encode the cleaned market column
from sklearn.preprocessing import LabelEncoder
le_market = LabelEncoder()
train_x['market'] = le_market.fit_transform(train_x['market'])

In [31]:
# Drop columns that are still object or datetime if not needed
train_x.drop(columns=['host_name'], inplace=True, errors='ignore')

# Convert datetime columns to numeric: year * 100 + month (or just timestamp)
for col in ['host_since', 'first_review']:
    if col in train_x.columns:
        train_x[col] = train_x[col].dt.year * 100 + train_x[col].dt.month


In [32]:
train_y['high_booking_rate'] = train_y['high_booking_rate'].map({'YES': 1, 'NO': 0})
train_y['perfect_rating_score'] = train_y['perfect_rating_score'].map({'YES': 1, 'NO': 0})

In [33]:
train_x.to_csv('train_x_final.csv', index=False)
train_y.to_csv('train_y_final.csv', index=False)

In [34]:
train_x = pd.read_csv('train_x_final.csv')
train_y = pd.read_csv('train_y_final.csv')

In [ ]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import numpy as np

# Step 1: Train-validation split
X_train, X_val, y_train, y_val = train_test_split(
    train_x, train_y, test_size=0.2, random_state=42
)

# Step 2: Define the base model
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

# Step 3: Define hyperparameter space
param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.3],
    'subsample': [0.5, 0.7, 0.8, 1.0],
    'colsample_bytree': [0.5, 0.7, 0.8, 1.0],
    'gamma': [0, 0.1, 0.3, 1],
    'reg_alpha': [0, 0.01, 0.1, 1],
    'reg_lambda': [0.5, 1, 1.5, 2]
}

# Step 4: Set up randomized search
random_search = RandomizedSearchCV(
    xgb_model,
    param_distributions=param_dist,
    n_iter=25,                  # Try 25 random combinations
    scoring='roc_auc',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1                   # Use all cores for speed
)

# Step 5: Train the model
random_search.fit(X_train, y_train)

# Step 6: Evaluate
best_model = random_search.best_estimator_
y_val_pred_proba = best_model.predict_proba(X_val)[:, 1]
val_auc = roc_auc_score(y_val, y_val_pred_proba)

print("Best parameters:", random_search.best_params_)
print("Validation AUC:", val_auc)


Fitting 3 folds for each of 25 candidates, totalling 75 fits


Validation AUC (XGBoost): 0.9037


In [ ]:
submission = submission.astype(str).str.replace(r"[\\n\\r]+", "", regex=True)

submission.to_csv("high_booking_rate_group5.csv", index=False)
print("high_booking_rate_group5.csv")


✅ Cleaned and saved: high_booking_rate_group5.csv
